In [1]:
from email import utils
import polars as pl
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import re
from tqdm import tqdm
sys.path.append("../src")  # Add path to where 'retail' package lives
import retail.retailfilter as rf # Correct import path

import utils.utils as ut  # Correct import path


def polars_to_geoparquet(
    pl_df: pl.DataFrame,
    out_path: str,
    lat_col: str = "lat",
    lng_col: str = "lng",
    crs: str = "EPSG:4326",
):
    pdf = pl_df.to_pandas()
    gdf = gpd.GeoDataFrame(
        pdf,
        geometry=gpd.points_from_xy(pdf[lng_col], pdf[lat_col]),
        crs=crs,
    )
    gdf.to_parquet(out_path, index=False)


[nltk_data] Downloading package names to /home/ogoodwin/nltk_data...
[nltk_data]   Package names is already up-to-date!


In [ ]:
df = pl.read_parquet("../data/POI_files/duplicate_analysis_full/duplicates_all_with_groups.parquet")
df

group_id,group_size,keep,row_id,fsq_place_id,name,lat,lng,address,tel,website,date_created,date_refreshed
i64,i64,bool,u32,str,str,f64,f64,str,str,str,str,str
683,63,true,1357986,"""fbe53acbfa4c4ecad88347c2""","""Westminster Locksmiths""",51.52433,-0.174952,"""18 Cooper House""","""020 3667 3601""","""https://locksmiths.ltd/south-w…","""2021-12-05""","""2024-06-23"""
683,63,false,616593,"""fa6bf60c5ecb457c3b707288""","""Streatham Hill Locksmiths SW2""",51.52433,-0.174952,"""18 Cooper House""","""020 3667 3601""","""https://locksmiths.ltd/streath…","""2022-01-16""","""2022-01-16"""
683,63,false,489567,"""927c55e05ad544fb098d8e54""","""24/7 Locksmith Services in Ken…",51.52433,-0.174952,"""18 Cooper House""","""020 3667 3601""","""https://locksmiths.ltd/kentish…","""2022-01-19""","""2022-01-19"""
683,63,false,1232651,"""f655927236174720e40be8cb""","""Locksmiths Wandsworth SW18""",51.52433,-0.174952,"""18 Cooper House""","""020 3667 3601""","""https://locksmiths.ltd/wandswo…","""2022-01-16""","""2022-01-16"""
683,63,false,259862,"""2ca2292aede24561b76fc30b""","""24/7 Locksmith Services in Eas…",51.52433,-0.174952,"""18 Cooper House""","""020 3667 3601""","""https://locksmiths.ltd/east-sh…","""2022-01-22""","""2022-01-22"""
…,…,…,…,…,…,…,…,…,…,…,…,…
7835,2,false,815346,"""5b530a64ee7120002c1625a9""","""Oliver Adams""",51.916774,-0.661552,"""7 Market Square""","""01525 377979""","""http://www.oliveradams.co.uk""","""2018-07-21""","""2020-04-07"""
7836,2,true,582182,"""5bd736d5c47cf9002ce08ac6""","""Payday Slots""",54.971843,-1.589535,"""Toffee Factory""","""07754 245302""","""https://www.dragonslots.co.uk""","""2018-10-29""","""2018-10-29"""
7836,2,false,494610,"""5bd5f3602be425002ca1823e""","""Dragon Slots""",54.971839,-1.589636,"""Toffee Factory""","""07754 245302""","""https://www.paydayslots.co.uk""","""2018-10-28""","""2018-10-28"""


: 

In [27]:
df = pl.read_parquet("../data/POI_files/duplicate_analysis_full/new_pairs_06_threshold.parquet")
df[["poi1_name", "poi2_name"]]

poi1_name,poi2_name
str,str
"""Lio’s""","""Lio"""
"""Bickerton Locksmiths""","""Shelley Locksmiths"""
"""PC's Cafe""","""Oasis Cafe"""
"""Acs""","""A C S"""
"""Bonnyin Schweiz""","""Bonnyin Austria"""
…,…
"""Kutz 4 kids""","""Kuts for Kids"""
"""Pauls Produce""","""Dunns Produce"""
"""Letchworth Central Library""","""Letchworth Library"""


In [3]:
input_file = Path("../data/input_files/uk_foursquare_cc_nov_iceberg.parquet")
# Read in parquet
foursquare = pl.read_parquet(input_file)

In [10]:
#get just foursquare["unresolved_flags"] aand not empyty list
foursquare_flags = foursquare.filter(
    pl.col("unresolved_flags").list.len() > 0
)

In [ ]:
foursquare_flags

fsq_place_id,name,lat,lng,address,locality,region,postcode,admin_region,post_town,po_box,country,date_created,date_refreshed,date_closed,tel,website,email,facebook_id,instagram,twitter,fsq_category_id,fsq_category_labels,unresolved_flags,geometry,h3_1,h3_2,h3_3,h3_4,h3_5,h3_6,h3_7,h3_8,h3_9,h3_10,h3_11,category_id,category_level,category_name,category_label,level1_category_id,level1_category_name,level2_category_id,level2_category_name,level3_category_id,level3_category_name,level4_category_id,level4_category_name,level5_category_id,level5_category_name,level6_category_id,level6_category_name
str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,list[str],list[str],binary,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""534808d7498ef3608ae7434b""","""Miracle Span Steel Buildings""",53.446256,0.131819,"""Units 4&5 South Road Ind Estat…","""North Somercotes, Louth""","""Lincs""","""LN11 7PT""","""England""",null,null,"""GB""","""2014-04-11""","""2014-05-22""",null,"""01507 358974""","""http://www.miraclespan.co.uk""",null,1.0794e14,null,"""miraclespan""","""52e81612bcbc57f1066b7a36""","[""Business and Professional Services > Warehouse""]","[""doesnt_exist""]","b""\x01\x01\x00\x00\x00E\xd0\xdf\xc5o\xdf\xc0?\x00\xf0\xe7\xe9\x1e\xb9J@""","""81197ffffffffff""","""821947fffffffff""","""831940fffffffff""","""8419407ffffffff""","""8519400ffffffff""","""861940767ffffff""","""871940764ffffff""","""881940764bfffff""","""891940764b7ffff""","""8a1940764b4ffff""","""8b1940764a64fff""","""52e81612bcbc57f1066b7a36""",2.0,"""Warehouse""","""Business and Professional Serv…","""4d4b7105d754a06375d81259""","""Business and Professional Serv…","""52e81612bcbc57f1066b7a36""","""Warehouse""",null,null,null,null,null,null,null,null
"""4c8a3721e51e6dcb19c066de""","""Smugglers Cave""",52.944994,1.213094,null,null,null,null,"""England""",null,null,"""GB""","""2010-09-10""","""2015-10-01""",null,"""01263 823069""",null,null,null,null,null,"""4bf58dd8d48988d128951735""","[""Retail > Gift Store""]","[""duplicate""]","b""\x01\x01\x00\x00\x00^p\xbc\xc2\xd5h\xf3?\xda\xc1\x9f\x8c\xf5xJ@""","""81197ffffffffff""","""821947fffffffff""","""831941fffffffff""","""8419411ffffffff""","""85194113fffffff""","""861941117ffffff""","""871941112ffffff""","""8819411129fffff""","""8919411129bffff""","""8a19411129affff""","""8b19411129aafff""","""4bf58dd8d48988d128951735""",2.0,"""Gift Store""","""Retail > Gift Store""","""4d4b7105d754a06378d81259""","""Retail""","""4bf58dd8d48988d128951735""","""Gift Store""",null,null,null,null,null,null,null,null
"""5200e16b498e022ef638b8e2""","""The Baguette Shop""",52.931183,1.299292,null,null,"""Norfolk""",null,"""England""",null,null,"""GB""","""2013-08-06""","""2022-11-21""",null,"""07748 272078""","""http://thebaguetteshopcromer.c…","""thebaguetteshop@btinternet.com""",null,null,null,"""4bf58dd8d48988d1c5941735""","[""Dining and Drinking > Restaurant > Sandwich Spot""]","[""duplicate""]","b""\x01\x01\x00\x00\x00\x00\x00\x00\x20\xe6\xc9\xf4?\x00\x00\x00\x001wJ@""","""81197ffffffffff""","""821947fffffffff""","""831941fffffffff""","""8419411ffffffff""","""85194113fffffff""","""861941107ffffff""","""871941106ffffff""","""8819411067fffff""","""89194110667ffff""","""8a194110665ffff""","""8b194110665dfff""","""4bf58dd8d48988d1c5941735""",3.0,"""Sandwich Spot""","""Dining and Drinking > Restaura…","""63be6904847c3692a84b9bb5""","""Dining and Drinking""","""4d4b7105d754a06374d81259""","""Restaurant""","""4bf58dd8d48988d1c5941735""","""Sandwich Spot""",null,null,null,null,null,null
"""4e5516fec65b9614d8ef026c""","""Regency Ice Cream""",52.93337,1.289837,null,"""Cromer""","""Norfolk""",null,"""England""",null,null,"""GB""","""2011-08-24""","""2012-05-31""",null,null,null,null,null,null,null,"""4bf58dd8d48988d1c9941735""","[""Dining and Drinking > Dessert Shop > Ice Cream Parlor""]","[""closed""]","b""\x01\x01\x00\x00\x00X\x04\xd0\x0e,\xa3\xf4?p\xd1\xf1\

: 

In [8]:
foursquare_flags

unresolved_flags
list[str]
[]
[]
[]
[]
[]
…
[]
[]
[]


In [2]:

# Input file
# input_file = Path("../data/input_files/uk_foursquare_datefields_admin_categories.parquet")
#nov data
input_file = Path("../data/input_files/uk_foursquare_cc_nov_iceberg.parquet")
# Read in parquet
foursquare = pl.read_parquet(input_file)
print(len(foursquare))
# Filter
foursquare_filtnew = rf.filter_foursquare(foursquare)

unique_levels = (
    foursquare_filtnew
    .group_by("level1_category_name", "level2_category_name", "level3_category_name")
    .len()
    .rename({"len": "count"})
    .sort("count", descending=True)
)

unique_levels.write_csv("../data/POI_files/foursquare_unique_levels.csv")
print(len(foursquare_filtnew))



# Columns to keep
COLUMNS_TO_KEEP = [
    "fsq_place_id", "name", "lat", "lng","fsq_category_labels", "address", "locality", "region",
    "postcode", "admin_region", "post_town", 
    "date_created", "date_refreshed", "date_closed", "tel", "website", 
    "fsq_category_ids", "h3_11","level1_category_name", "level2_category_name", "level3_category_name",
]

# name has changed now its exploded.
if "fsq_category_ids" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_id" else "fsq_category_ids" for col in COLUMNS_TO_KEEP]
elif "fsq_category_id" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_ids" else "fsq_category_id" for col in COLUMNS_TO_KEEP]
else:
    COLUMNS_TO_KEEP_MOD = COLUMNS_TO_KEEP

# Add H3 column
foursquare_filtnew = ut.add_h3_11_column(foursquare_filtnew)

# Keep only selected columns
foursquare_filtnew = foursquare_filtnew.select(COLUMNS_TO_KEEP_MOD)





4695350
Filtered Foursquare data (Retail Cats): 1938605 rows remaining.
Filtered Foursquare data (remove post box): 1938552 rows remaining.
Filtered Foursquare data (remove closed): 1666321 rows remaining.
Filtered Foursquare data (remove cat dupes): 1499296 rows remaining.
1499296


In [10]:
import math
import polars as pl
import h3  # pip install h3

EARTH_R_M = 6_371_000.0
dist_thresh = 15
# --- helpers from earlier (unchanged) ---
def _norm_text(s: str) -> str:
    if s is None:
        return ""
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _norm_site(s: str) -> str:
    if not s:
        return ""
    s = s.lower().strip()
    s = re.sub(r"^https?://(www\.)?", "", s)
    s = s.rstrip("/")
    return s

def _levenshtein(a: str, b: str) -> int:
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    if len(a) > len(b): a, b = b, a
    prev = list(range(len(a) + 1))
    for j, cb in enumerate(b, start=1):
        curr = [j]
        for i, ca in enumerate(a, start=1):
            curr.append(min(prev[i] + 1, curr[i - 1] + 1, prev[i - 1] + (ca != cb)))
        prev = curr
    return prev[-1]

def _lev_ratio(a: str, b: str) -> float:
    la, lb = len(a), len(b)
    if la == 0 and lb == 0:
        return 0.0
    return 1.0 - (_levenshtein(a, b) / max(la, lb))

def _haversine_m(lat1, lon1, lat2, lon2) -> float:
    # Accurate enough for 5 m cutoff
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = phi2 - phi1
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * EARTH_R_M * math.asin(math.sqrt(a))

name_threshold = 0.5

# ---- base df: add row_id only once ----
df = foursquare_filtnew
if "row_id" not in df.columns:
    df = df.with_row_index("row_id")

# ---- normalized text/site columns (keep originals for distance) ----
dfn = (
    df.with_columns(
        pl.col("lat").cast(pl.Float64),
        pl.col("lng").cast(pl.Float64),
        pl.col("name").map_elements(_norm_text, return_dtype=str).alias("name_norm"),
        pl.col("address").map_elements(_norm_text, return_dtype=str).alias("address_norm"),
        pl.col("tel").map_elements(_norm_text, return_dtype=str).alias("tel_norm"),
        pl.col("website").map_elements(_norm_site, return_dtype=str).alias("website_norm"),
    )
    # keep only the columns we will need going forward to save memory
    .select([
        "row_id", "lat", "lng", "h3_11", "fsq_category_id",
        "name_norm", "address_norm", "tel_norm", "website_norm"
    ])
)

# ---- cell -> neighbors (k=1 ring) mapping (includes the cell itself) ----
# Build once based on the unique cells present
unique_cells = dfn.select("h3_11").unique()

neighbors = (
    unique_cells
    .with_columns(
        pl.col("h3_11")
          .map_elements(lambda h: list(h3.grid_disk(h, 1)), return_dtype=pl.List(pl.Utf8))
          .alias("cand_cells")
    )
    .explode("cand_cells")
    .rename({"h3_11": "src_cell", "cand_cells": "cand_cell"})
)

# ---- Expand each row to its 7 candidate cells (self + 6 neighbors) ----
dfn_exp = (
    dfn.join(neighbors, left_on="h3_11", right_on="src_cell", how="inner")
       .select([
           "row_id", "lat", "lng", "h3_11", "fsq_category_id",
           "name_norm", "address_norm", "tel_norm", "website_norm",
           "cand_cell"
       ])
)

pairs_raw = (
    dfn_exp.join(
        dfn,
        left_on=["cand_cell", "fsq_category_id"],
        right_on=["h3_11", "fsq_category_id"],
        how="inner",
        suffix="_r",
    )
    .with_columns(pl.col("cand_cell").alias("h3_11_r"))   # <-- add this
    .filter(pl.col("row_id") < pl.col("row_id_r"))
)


# ---- (cheap) pre-filter by tiny bbox to trim far pairs before exact distance (optional but helpful) ----
# Using conservative bounds so we don't drop true 5m matches.
pairs_pre = pairs_raw.filter(
    ((pl.col("lat") - pl.col("lat_r")).abs() <= 1) &   # ~<= 16-17 m in latitude
    ((pl.col("lng") - pl.col("lng_r")).abs() <= 1)      # generous in longitude to be safe across latitudes
)

# ---- Exact distance (Haversine) + original similarity logic ----
pairs = (
    pairs_pre
    .with_columns(
        pl.struct(["lat","lng","lat_r","lng_r"])
          .map_elements(lambda s: _haversine_m(s["lat"], s["lng"], s["lat_r"], s["lng_r"]), return_dtype=pl.Float64)
          .alias("dist_m"),
        (pl.col("address_norm") == pl.col("address_norm_r")).alias("m_address"),
        (pl.col("tel_norm") == pl.col("tel_norm_r")).alias("m_tel"),
        (pl.col("website_norm") == pl.col("website_norm_r")).alias("m_website"),
        pl.struct(["name_norm","name_norm_r"])
          .map_elements(lambda s: _lev_ratio(s["name_norm"], s["name_norm_r"]), return_dtype=pl.Float64)
          .alias("name_sim"),
    )
    .with_columns((pl.col("name_sim") >= name_threshold).alias("m_name"))
    # Now require BOTH: within 5 m AND (any of the other fields match)
    .with_columns(
        ((pl.col("dist_m") <= dist_thresh) & (pl.col("m_address") | pl.col("m_tel") | pl.col("m_website") | pl.col("m_name")))
        .alias("is_match")
    )
    .filter(pl.col("is_match"))
    .with_columns(
        pl.concat_list([
            pl.when(pl.col("m_address")).then(pl.lit("address")).otherwise(None),
            pl.when(pl.col("m_tel")).then(pl.lit("tel")).otherwise(None),
            pl.when(pl.col("m_website")).then(pl.lit("website")).otherwise(None),
            pl.when(pl.col("m_name")).then(pl.lit("name")).otherwise(None),
            pl.when(pl.col("dist_m") < dist_thresh).then(pl.lit("≤5m")).otherwise(None),
        ]).alias("match_list_raw")
    )
    .with_columns(pl.col("match_list_raw").list.drop_nulls().alias("match_list"))
    .select(
        "row_id", "row_id_r",
        "h3_11", pl.col("h3_11_r"),
        "fsq_category_id",
        "dist_m",
        "match_list"
    )
)

# ---- implicated row_ids ----
implicated_ids = pl.concat([
    pairs.select(pl.col("row_id").alias("row_id")),
    pairs.select(pl.col("row_id_r").alias("row_id")),
]).unique()

# ---- per-row union of matched fields ----
row_match_fields = (
    pl.concat([
        pairs.select("row_id", "match_list").rename({"row_id": "rid"}),
        pairs.select(pl.col("row_id_r").alias("rid"), "match_list"),
    ])
    .group_by("rid")
    .agg(pl.col("match_list").flatten().alias("match_fields_list"))
    .with_columns(
        pl.col("match_fields_list").list.unique().list.sort().list.join(", ").alias("match_fields")
    )
    .select(pl.col("rid").alias("row_id"), "match_fields")
)

# ---- per-row match degree (how many within-5m matches this row has) ----
dup_degree = (
    pl.concat([
        pairs.select(pl.col("row_id").alias("rid")),
        pairs.select(pl.col("row_id_r").alias("rid")),
    ])
    .group_by("rid")
    .agg((pl.len() + 1).alias("dup_count"))
    .rename({"rid": "row_id"})
)

# ---- final rows, sorted by how "duplicate-y" they are ----
dupes_grouped = (
    dfn.join(implicated_ids, on="row_id", how="inner")
       .join(dup_degree, on="row_id", how="left")
       .join(row_match_fields, on="row_id", how="left")
       .sort(["dup_count", "h3_11"], descending=[True, False])
)

dupes_grouped


row_id,lat,lng,h3_11,fsq_category_id,name_norm,address_norm,tel_norm,website_norm,dup_count,match_fields
u32,f64,f64,str,str,str,str,str,str,u32,str
546340,51.52433,-0.174952,"""8b195da48782fff""","""52f2ab2ebcbc57f1066b8b1e""","""locksmith services in earls co…","""18 cooper house""","""020 3667 3601""","""locksmiths.ltd/earls-court-loc…",63,"""address, name, tel, ≤5m"""
726862,51.52433,-0.174952,"""8b195da48782fff""","""52f2ab2ebcbc57f1066b8b1e""","""24 7 locksmith services in ken…","""18 cooper house""","""020 3667 3601""","""locksmiths.ltd/kentish-town-lo…",63,"""address, name, tel, ≤5m"""
1171814,51.52433,-0.174952,"""8b195da48782fff""","""52f2ab2ebcbc57f1066b8b1e""","""24 7 locksmith services in gol…","""18 cooper house""","""020 3667 3601""","""locksmiths.ltd/golders-green-l…",63,"""address, name, tel, website, ≤…"
1106246,51.52433,-0.174952,"""8b195da48782fff""","""52f2ab2ebcbc57f1066b8b1e""","""24 7 locksmith services in cam…","""18 cooper house""","""020 3667 3601""","""locksmiths.ltd/camden-town-loc…",63,"""address, name, tel, ≤5m"""
140453,51.52433,-0.174952,"""8b195da48782fff""","""52f2ab2ebcbc57f1066b8b1e""","""cricklewood locksmiths nw2""","""18 cooper house""","""020 3667 3601""","""locksmiths.ltd/cricklewood-loc…",63,"""address, name, tel, ≤5m"""
…,…,…,…,…,…,…,…,…,…,…
200869,57.307473,-2.626802,"""8b1976c40628fff""","""4bf58dd8d48988d128951735""","""ingrained inc""","""the smiddy""","""07507 555218""",null,2,"""address, tel, ≤5m"""
1110168,57.444712,-3.127037,"""8b1976c94351fff""","""4bf58dd8d48988d11b951735""","""floral occasions""","""15 fife st""","""01340 820567""","""dufftownfloraloccasions.co.uk""",2,"""tel, ≤5m"""
274188,57.444704,-3.127252,"""8b1976c94353fff""","""4bf58dd8d48988d11b951735""","""rustic roses""","""7 fife st""","""01340 820567""","""rustic-and-roses.co.uk""",2,"""tel, ≤5m"""


In [24]:
#just dupes_grouped where match_fields contains only contains address and ≤5m

dupes_grouped_filtered = dupes_grouped.filter(
    (pl.col("match_fields") == "address, ≤5m")
)

In [25]:

dupes_grouped_filtered

row_id,lat,lng,h3_11,fsq_category_id,name_norm,address_norm,tel_norm,website_norm,dup_count,match_fields
u32,f64,f64,str,str,str,str,str,str,u32,str
145188,57.180641,-2.152922,"""8b19768ccd8cfff""","""52f2ab2ebcbc57f1066b8b28""","""canon imageclass mf644cdw wire…","""14502 north dale mabry highway…","""0844 898 6210""","""all-printer-setup.com/canon-im…",27,"""address, ≤5m"""
989838,57.180641,-2.152922,"""8b19768ccd8cfff""","""52f2ab2ebcbc57f1066b8b28""","""hp officejet pro 8615 driver d…","""14502 north dale mabry highway…","""0844 788 0600""","""b123-hp.com/123-hp-com-ojpro86…",27,"""address, ≤5m"""
454546,57.180641,-2.152922,"""8b19768ccd8cfff""","""52f2ab2ebcbc57f1066b8b28""","""hp eprint setup""","""14502 north dale mabry highway…","""07824 859307""","""hpeprint.us""",27,"""address, ≤5m"""
389704,51.514806,-0.123674,"""8b194ad32436fff""","""4d4b7105d754a06378d81259""","""marcus rose international""","""71 75 shelton st""","""07802 709974""","""marcusroseinternational.com""",24,"""address, ≤5m"""
177422,51.514844,-0.123652,"""8b194ad32436fff""","""4d4b7105d754a06378d81259""","""euro matic uk""","""71 75 shelton st""","""0330 311 0003""","""euro-matic.co.uk""",23,"""address, ≤5m"""
…,…,…,…,…,…,…,…,…,…,…
1032257,57.146921,-2.100674,"""8b1976bb2d1dfff""","""4bf58dd8d48988d110951735""","""de milo""","""3 little belmont st""","""01224 640222""",null,2,"""address, ≤5m"""
274908,57.146233,-2.104644,"""8b1976bb2d89fff""","""4bf58dd8d48988d11e941735""","""bonne nuit""","""8 golden square""",null,null,2,"""address, ≤5m"""
957686,57.146233,-2.104627,"""8b1976bb2d89fff""","""4bf58dd8d48988d11e941735""","""barbelow""","""8 golden square""",null,null,2,"""address, ≤5m"""


In [ ]:

# #run with level 2 
# # use rounded coords in the duplicate key
# keys = ["lat_round", "lng_round", "level2_category_name"]

# pairs = (
#     dfn.join(dfn, on=keys, how="inner", suffix="_r")
#        .filter(pl.col("row_id") < pl.col("row_id_r"))
#        # exclude pairs with identical fsq_category_id
#        .filter(pl.col("fsq_category_id") != pl.col("fsq_category_id_r"))
#        .with_columns(
#            (pl.col("address_norm") == pl.col("address_norm_r")).alias("m_address"),
#            (pl.col("tel_norm") == pl.col("tel_norm_r")).alias("m_tel"),
#            (pl.col("website_norm") == pl.col("website_norm_r")).alias("m_website"),
#            pl.struct(["name_norm","name_norm_r"])
#              .map_elements(lambda s: _lev_ratio(s["name_norm"], s["name_norm_r"]))
#              .alias("name_sim"),
#        )
#        .with_columns((pl.col("name_sim") >= name_threshold).alias("m_name"))
#        .with_columns(
#            (pl.col("m_address") | pl.col("m_tel") | pl.col("m_website") | pl.col("m_name")).alias("is_match")
#        )
#        .filter(pl.col("is_match"))
#        .with_columns(
#            pl.concat_list([
#                pl.when(pl.col("m_address")).then(pl.lit("address")).otherwise(None),
#                pl.when(pl.col("m_tel")).then(pl.lit("tel")).otherwise(None),
#                pl.when(pl.col("m_website")).then(pl.lit("website")).otherwise(None),
#                pl.when(pl.col("m_name")).then(pl.lit("name")).otherwise(None),
#            ]).alias("match_list_raw")
#        )
#        .with_columns(pl.col("match_list_raw").list.drop_nulls().alias("match_list"))
#        .select("row_id", "row_id_r", *keys, "match_list")
# )

# # implicated row_ids
# implicated_ids = (
#     pl.concat([
#         pairs.select(pl.col("row_id").alias("row_id")),
#         pairs.select(pl.col("row_id_r").alias("row_id")),
#     ])
#     .unique()
# )

# # per-row union of matched fields
# row_match_fields = (
#     pl.concat([
#         pairs.select("row_id", "match_list").rename({"row_id": "rid"}),
#         pairs.select(pl.col("row_id_r").alias("rid"), "match_list"),
#     ])
#     .group_by("rid")
#     .agg(pl.col("match_list").flatten().alias("match_fields_list"))
#     .with_columns(
#         pl.col("match_fields_list").list.unique().list.sort().list.join(", ").alias("match_fields")
#     )
#     .select(pl.col("rid").alias("row_id"), "match_fields")
# )

# # add dup_count per rounded-key group and keep groups together, largest first
# dupes_grouped = (
#     dfn.join(implicated_ids, on="row_id", how="inner")   # <-- use dfn (has lat_round/lng_round)
#       .join(
#           implicated_ids.join(dfn.select("row_id", *keys), on="row_id", how="inner")  # <-- dfn here too
#                         .group_by(keys).agg(pl.len().alias("dup_count")),
#           on=keys, how="left"
#       )
#       .join(row_match_fields, on="row_id", how="left")
#       .sort(["dup_count"] + keys, descending=[True, False, False, False])
# )

# #just level 2 matches which dont match cats. if we look this doesnt actually work very well, removes lots of real places.
# dupes_grouped


/tmp/ipykernel_116950/370397565.py:4: DeprecationWarning: `DataFrame.with_row_count` is deprecated. Use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  df = df.with_row_count("row_id")
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_eleme

row_id,fsq_place_id,name,lat,lng,fsq_category_labels,address,locality,region,postcode,admin_region,post_town,date_created,date_refreshed,date_closed,tel,website,fsq_category_id,h3_11,level1_category_name,level2_category_name,level3_category_name,lat_round,lng_round,name_norm,address_norm,tel_norm,website_norm,dup_count,match_fields
u32,str,str,f64,f64,list[str],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,u32,str
1403453,"""892635a4dd9c4dbf2966ac3d""","""Members.Marssociety.Org/Member…",51.428651,-0.349189,"[""Health and Medicine > Healthcare Clinic"", ""Retail > Food and Beverage Retail > Health Food Store""]","""3 Oxford Rd""","""Teddington""","""Greater London""","""TW11 0QA""","""England""","""Teddington""","""2022-01-12""","""2022-01-12""",null,"""020 7412 8988""",null,"""50aa9e744b90af0d42d5de0e""","""8b194ad9e106fff""","""Retail""","""Food and Beverage Retail""","""Health Food Store""",51.428651,-0.349189,"""members marssociety org member…","""3 oxford rd""","""020 7412 8988""",null,35,"""address, tel"""
973922,"""841fc96be39b42b9100228c6""","""603210F6D56A2.Site123.Me""",51.428651,-0.349189,"[""Retail > Food and Beverage Retail > Health Food Store""]","""3 Oxford Rd""","""Teddington""","""Greater London""","""TW11 0QA""","""England""","""Teddington""","""2021-12-25""","""2021-12-25""",null,"""020 7412 8988""","""https://603210f6d56a2.site123.…","""50aa9e744b90af0d42d5de0e""","""8b194ad9e106fff""","""Retail""","""Food and Beverage Retail""","""Health Food Store""",51.428651,-0.349189,"""603210f6d56a2 site123 me""","""3 oxford rd""","""020 7412 8988""","""603210f6d56a2.site123.me""",35,"""address, tel"""
775056,"""6fd5126d27ea421f3d258def""","""v.gd/wHAqfh""",51.428651,-0.349189,"[""Retail > Food and Beverage Retail > Health Food Store""]","""3 Oxford Rd""","""Teddington""","""Greater London""","""TW11 0QA""","""England""","""Teddington""","""2021-12-12""","""2021-12-12""",null,"""020 7412 8988""","""https://v.gd/wHAqfh""","""50aa9e744b90af0d42d5de0e""","""8b194ad9e106fff""","""Retail""","""Food and Beverage Retail""","""Health Food Store""",51.428651,-0.349189,"""v gd whaqfh""","""3 oxford rd""","""020 7412 8988""","""v.gd/whaqfh""",35,"""address, tel"""
869315,"""58559fbd5d3f43eff1075063""","""Blogs.Deusto.Es/Aplicaciones-T…",51.428651,-0.349189,"[""Health and Medicine > Healthcare Clinic"", ""Retail > Food and Beverage Retail > Health Food Store""]","""3 Oxford Rd""","""Teddington""","""Greater London""","""TW11 0QA""","""England""","""Teddington""","""2022-01-09""","""2022-01-09""",null,"""020 7412 8988""","""https://bibliotecadulcenombre.…","""50aa9e744b90af0d42d5de0e""","""8b194ad9e106fff""","""Retail""","""Food and Beverage Retail""","""Health Food Store""",51.428651,-0.349189,"""blogs deusto es aplicaciones t…","""3 oxford rd""","""020 7412 8988""","""bibliotecadulcenombre.es.tl/bi…",35,"""address, tel"""
770426,"""74085726467545faa38af8d5""","""Scienceday.Bio.Upenn.Edu/Feedb…",51.428651,-0.349189,"[""Retail > Food and Beverage Retail > Health Food Store""]","""3 Oxford Rd""","""Teddington""","""Greater London""","""TW11 0QA""","""England""","""Teddington""","""2022-01-15""","""2022-01-15""",null,"""020 7412 8988""",null,"""50aa9e744b90af0d42d5de0e""","""8b194ad9e106fff""","""Retail""","""Food and Beverage Retail""","""Health Food Store""",51.428651,-0.349189,"""scienceday bio upenn edu feedb…","""3 oxford rd""","""020 7412 8988""",null,35,"""address, tel"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
332133,"""4d8615a999b78cfa4264dc1f""","""Forres Swimming Pool and Fitne…",57.604425,-3.611392,"[""Sports and Recreation > Gym and Studio > Gym""]",null,null,null,null,"""Scotland""",null,"""2011-03-20""","""2019-08-09""",null,"""01309 672984""","""http://www.moray.gov.uk/moray_…","""4bf58dd8d48988d176941735""","""8b192b38c774fff""","""Sports and Recreation""","""Gym and Studio""","""Gym""",57.604425,-3.611392,"""forres swimming pool and fitne…",null,"""01309

In [ ]:
import polars as pl

date_col = "date_created"  # <-- string column

# 1) Parse the string into a datetime column
dupes_with_date = (
    dupes_grouped.with_columns(
        pl.col(date_col)
          .str.strptime(pl.Datetime, strict=False, exact=False)
          .alias("date_ts")
    )
)

# 2) Pick the newest row per (lat,lng,fsq_category_id) group
keep_ids = (
    dupes_with_date
    .sort(["date_ts", "row_id"], descending=[True, False], nulls_last=True)
    .group_by(["lat_round", "lng_round", "fsq_category_id"])
    .agg(pl.first("row_id").alias("keep_row_id"))
    .select(pl.col("keep_row_id").alias("row_id"))
)

# 3) Rows to remove (older dupes)
rows_to_remove = dupes_grouped.join(keep_ids, on="row_id", how="anti")

# 4) Apply to the full dataset: keep newest only
df_dedup_latest = df.join(rows_to_remove.select("row_id"), on="row_id", how="anti")

# 5) Report how many places were removed
print(f"Removed {rows_to_remove.height} places by keeping only the most recent date_created per duplicate group.")

# Optional: see which rows were kept from the dupes
kept_from_dupes = dupes_grouped.join(keep_ids, on="row_id", how="inner")
kept_from_dupes


In [4]:
import polars as pl

# US states
US_STATE_CODES = [
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","IA","ID","IL","IN","KS","KY","LA",
    "MA","MD","ME","MI","MN","MO","MS","MT","NC","ND","NE","NH","NJ","NM","NV","NY","OH","OK",
    "OR","PA","RI","SC","SD","TN","TX","UT","VA","VT","WA","WI","WV","DC"
]
US_STATE_NAMES = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut","Delaware",
    "Florida","Georgia","Hawaii","Iowa","Idaho","Illinois","Indiana","Kansas","Kentucky",
    "Louisiana","Massachusetts","Maryland","Maine","Michigan","Minnesota","Missouri","Mississippi",
    "Montana","North Carolina","North Dakota","Nebraska","New Hampshire","New Jersey","New Mexico",
    "Nevada","New York","Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina",
    "South Dakota","Tennessee","Texas","Utah","Virginia","Vermont","Washington","Wisconsin",
    "West Virginia","District of Columbia"
]

# Canada
CA_PROV_CODES = ["AB","BC","MB","NB","NL","NS","NT","NU","ON","PE","QC","SK","YT"]
CA_PROV_NAMES = [
    "Alberta","British Columbia","Manitoba","New Brunswick","Newfoundland and Labrador","Nova Scotia",
    "Northwest Territories","Nunavut","Ontario","Prince Edward Island","Quebec","Saskatchewan","Yukon"
]

# Australia
AU_STATE_CODES = ["NSW","VIC","QLD","SA","WA","TAS","ACT","NT"]
AU_STATE_NAMES = [
    "New South Wales","Victoria","Queensland","South Australia","Western Australia","Tasmania",
    "Australian Capital Territory","Northern Territory"
]

# country mentions
COUNTRY_WORDS = [
    r"United\s+States", r"U\.?S\.?A\.?", r"Canada", r"France", r"Germany", r"Deutschland", r"Spain",
    r"España", r"Italy", r"Italia", r"Australia", r"Ireland", r"Republic\s+of\s+Ireland",
    r"Netherlands", r"Nederland", r"Belgium", r"Belgique", r"Luxembourg", r"Switzerland", r"Suisse",
    r"Sweden", r"Norway", r"Denmark", r"Finland", r"Austria", r"Portugal"
]

# distinctive postal formats
PAT_US_ZIP_END = r"\d{5}(?:-\d{4})?$"
PAT_CA_PC_END  = r"[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z]\s?\d[ABCEGHJ-NPRSTV-Z]\d$"
PAT_NL_PC_END  = r"\d{4}\s?[A-Z]{2}$"
PAT_IE_EIR_END = r"[A-Z]\d{2}\s?[A-Z0-9]{4}$"

# regexes
PAT_US_STATE_TAIL      = rf"(?:,?\s*)\b({'|'.join(US_STATE_CODES)})\b(?:\s*{PAT_US_ZIP_END})?$"        # CAPS only
PAT_US_STATE_NAME_TAIL = rf"(?i)(?:,?\s*)\b({'|'.join(US_STATE_NAMES)})\b(?:\s*{PAT_US_ZIP_END})?$"

PAT_CA_PROV_TAIL       = rf"(?:,?\s*)\b({'|'.join(CA_PROV_CODES)})\b(?:\s*{PAT_CA_PC_END})?$"          # CAPS only
PAT_CA_PROV_NAME_TAIL  = rf"(?i)(?:,?\s*)\b({'|'.join(CA_PROV_NAMES)})\b(?:\s*{PAT_CA_PC_END})?$"

PAT_AU_STATE_TAIL      = rf"(?:,?\s*)\b({'|'.join(AU_STATE_CODES)})\b(?:\s*\d{{4}})?$"                 # CAPS only
PAT_AU_STATE_NAME_TAIL = rf"(?i)(?:,?\s*)\b({'|'.join(AU_STATE_NAMES)})\b(?:\s*\d{{4}})?$"

PAT_COUNTRY_WORDS      = rf"(?i)\b({'|'.join(COUNTRY_WORDS)})\b"

# safe pass for Northern Ireland
SAFE_NI = r"(?i)\bNorthern\s+Ireland\b"


# keep if postcode is present
has_postcode = pl.col("postcode").is_not_null()

# foreign indicator from before (state codes, country names, etc.)
foreign_indicator = (
    pl.col("address").str.contains(PAT_COUNTRY_WORDS, literal=False)
    | pl.col("address").str.contains(PAT_US_STATE_TAIL, literal=False)
    | pl.col("address").str.contains(PAT_US_STATE_NAME_TAIL, literal=False)
    | pl.col("address").str.contains(PAT_CA_PROV_TAIL, literal=False)
    | pl.col("address").str.contains(PAT_CA_PROV_NAME_TAIL, literal=False)
    | pl.col("address").str.contains(PAT_AU_STATE_TAIL, literal=False)
    | pl.col("address").str.contains(PAT_AU_STATE_NAME_TAIL, literal=False)
    | pl.col("address").str.contains(PAT_CA_PC_END, literal=False)
    | pl.col("address").str.contains(PAT_NL_PC_END, literal=False)
    | pl.col("address").str.contains(PAT_IE_EIR_END, literal=False)
) & (~pl.col("address").str.contains(SAFE_NI, literal=False))



# Apply the filter
non_uk = foursquare_filtnew.filter(foreign_indicator & ~has_postcode)


In [ ]:
import polars as pl

date_col = "date_created"  # <-- string column

# 1) Parse the string into a datetime column
dupes_with_date = (
    dupes_grouped.with_columns(
        pl.col(date_col)
          .str.strptime(pl.Datetime, strict=False, exact=False)
          .alias("date_ts")
    )
)

# 2) Pick the newest row per (lat,lng,fsq_category_id) group
keep_ids = (
    dupes_with_date
    .sort(["date_ts", "row_id"], descending=[True, False], nulls_last=True)
    .group_by(["lat_round", "lng_round", "fsq_category_id"])
    .agg(pl.first("row_id").alias("keep_row_id"))
    .select(pl.col("keep_row_id").alias("row_id"))
)

# 3) Rows to remove (older dupes)
rows_to_remove = dupes_grouped.join(keep_ids, on="row_id", how="anti")

# 4) Apply to the full dataset: keep newest only
df_dedup_latest = df.join(rows_to_remove.select("row_id"), on="row_id", how="anti")

# 5) Report how many places were removed
print(f"Removed {rows_to_remove.height} places by keeping only the most recent date_created per duplicate group.")

# Optional: see which rows were kept from the dupes
kept_from_dupes = dupes_grouped.join(keep_ids, on="row_id", how="inner")
kept_from_dupes


Removed 9894 places by keeping only the most recent date_created per duplicate group.


row_id,fsq_place_id,name,lat,lng,fsq_category_labels,address,locality,region,postcode,admin_region,post_town,date_created,date_refreshed,date_closed,tel,website,fsq_category_id,h3_11,lat_round,lng_round,name_norm,address_norm,tel_norm,website_norm,dup_count,match_fields
u32,str,str,f64,f64,list[str],str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,u32,str
5593,"""66d69f67f01e4c865e9d6c81""","""24/7 Locksmith Services in Nor…",51.52433,-0.174952,"[""Business and Professional Services > Locksmith""]","""18 Cooper House""","""London""","""Greater London""","""NW8 8NJ""","""England""","""London""","""2022-01-22""","""2022-01-22""",null,"""020 3667 3601""","""https://locksmiths.ltd/locksmi…","""52f2ab2ebcbc57f1066b8b1e""","""8b195da48782fff""",51.52433,-0.174952,"""24 7 locksmith services in nor…","""18 cooper house""","""020 3667 3601""","""locksmiths.ltd/locksmith-near-…",63,"""address, name, tel"""
8146,"""f0c2585fa0824ba53e6c6682""","""revistas.uned.es/index.php/REE…",51.428651,-0.349189,"[""Retail > Food and Beverage Retail > Health Food Store""]","""3 Oxford Rd""","""Teddington""","""Greater London""","""TW11 0QA""","""England""","""Teddington""","""2021-12-22""","""2021-12-22""",null,"""020 7412 8988""","""http://revistas.uned.es/index.…","""50aa9e744b90af0d42d5de0e""","""8b194ad9e106fff""",51.428651,-0.349189,"""revistas uned es index php ree…","""3 oxford rd""","""020 7412 8988""","""revistas.uned.es/index.php/ree…",33,"""address, tel"""
111495,"""7acf3556669d411e803cb8e7""","""Get Drivers for HP Envy 4500 p…",57.180641,-2.152922,"[""Business and Professional Services > Telecommunication Service"", ""Retail > Print Store""]","""North Dale Mabry Highway, Para…",null,null,null,null,null,"""2022-01-07""","""2022-01-07""",null,"""0844 608 1510""","""https://www.hpenvy4500.com/how…","""52f2ab2ebcbc57f1066b8b28""","""8b19768ccd8cfff""",57.180641,-2.152922,"""get drivers for hp envy 4500 p…","""north dale mabry highway param…","""0844 608 1510""","""hpenvy4500.com/how-to-fix-hp-e…",29,"""tel"""
108359,"""6fda33e83f0146ba0e48f255""","""Biglycan Elisa""",56.127508,-3.138937,"[""Sports and Recreation > Gym and Studio""]","""45-1 Ramsey Rd""","""Shirley""","""Derbyshire""",null,"""England""",null,"""2022-01-09""","""2022-01-09""",null,"""020 1754 8553""","""http://www.creative-diagnostic…","""4bf58dd8d48988d175941735""","""8b197201d260fff""",56.127508,-3.138937,"""biglycan elisa""","""45 1 ramsey rd""","""020 1754 8553""","""creative-diagnostics.com/symbo…",23,"""address, tel"""
54861,"""b5e0218083ca4652830ee5ec""","""Cell Cycle Assays""",56.127508,-3.138937,"[""Retail > Medical Supply Store""]","""45-1 Ramsey Rd""","""New York""","""Lincolnshire""",null,"""England""",null,"""2022-01-15""","""2022-01-15""",null,"""020 7048 3343""","""https://www.creative-bioarray.…","""58daa1558bbb0b01f18ec206""","""8b197201d260fff""",56.127508,-3.138937,"""cell cycle assays""","""45 1 ramsey rd""","""020 7048 3343""","""creative-bioarray.com/cell-cyc…",23,"""address, tel"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
766972,"""90ff78dea7d74c8eeeaa7794""","""Talmine Post Office""",58.521178,-4.425628,"[""Community and Government > Government Building > Post Office""]","""The Post Office, Sinnet""","""Skinnet""","""Sutherland""","""IV27 4YP""","""Scotland""","""Lairg""","""2010-05-18""","""2010-05-18""",null,"""01847 601200""",null,"""4bf58dd8d48988d172941735""","""8b192aaf01aafff""",58.521178,-4.425628,"""talmine post office""","""the post office sinnet""","""01847 601200""",null,2,"""name"""
61564,"""bdd2213fd76648dc084ed378""","""Post Office""",58.594557,-3.518841,"[""Community and Government > Government Building > Post Office""]","""Scottish Co-Op""","""Thurso""","""Highland""","""KW14 8ER""","""Scotland""","""Thurso""","""2013-09-10""","""2025-08-04""",null,"""0345 722 3344""","""http://www.postoffice.co.uk""","""4bf58dd8d48988d172941735""","""8b192871d8f3fff""",58.594557,-3.518841,"""post office""","""scotti

In [7]:
foursquare_filtnew
#find mccolls
mccolls = foursquare_filtnew.filter(
    pl.col("name").str.contains(r"(?i)\bmccoll'?s\b", literal=False)
)
mccolls

fsq_place_id,name,lat,lng,fsq_category_labels,address,locality,region,postcode,admin_region,post_town,date_created,date_refreshed,date_closed,tel,website,fsq_category_id,h3_11,level1_category_name,level2_category_name,level3_category_name
str,str,f64,f64,list[str],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""5ea32f64220b2f0007f0fb07""","""Mccoll's Brewery""",54.61864,-1.757924,"[""Dining and Drinking > Brewery""]",null,"""Evenwood""","""England""","""DL14 9QA""","""England""",null,"""2020-04-24""","""2023-08-26""",null,"""01388 417250""","""http://www.mccollsbrewery.co.u…","""50327c8591d4c4b30a586d5d""","""8b1946486c60fff""","""Dining and Drinking""","""Brewery""",null
"""68a51f5cabf46a6415e874d5""","""McColl's""",56.00421,-4.734106,"[""Retail > Food and Beverage Retail > Grocery Store""]","""40 West Princes St""","""Helensburgh""",null,"""G84 8TD""",null,null,"""2025-08-20""","""2025-08-20""",null,"""01436 678919""",null,"""4bf58dd8d48988d118951735""","""8b1909071d49fff""","""Retail""","""Food and Beverage Retail""","""Grocery Store"""
"""79276febe95c4b4980fe33c9""","""Scholar Green McColls""",53.109379,-2.252447,"[""Retail""]","""222 Congleton Rd North""","""Scholar Green""","""Cheshire""","""ST7 3HF""","""England""",null,"""2021-11-16""","""2021-11-16""",null,"""01782 785905""",null,"""4d4b7105d754a06378d81259""","""8b195cc34120fff""","""Retail""",null,null
"""e7dd573372a24f1b52ad57fe""","""Mccoll's Retail Group""",53.419782,-2.724076,"[""Retail > Convenience Store""]","""The Shopping Centre, Four Acre…","""Clock Face""","""Merseyside""","""WA9 4BZ""","""England""","""Saint Helens""","""2019-07-17""","""2019-07-17""",null,"""01744 832824""",null,"""4d954b0ea243a5684a65b473""","""8b1951a86980fff""","""Retail""","""Convenience Store""",null
"""4caf50d875ebb60c083fbbad""","""McColls News Scotch Estate""",54.963592,-1.465734,"[""Retail > Food and Beverage Retail > Candy Store""]","""Edinburgh Road""","""Jarrow""",null,null,"""England""",null,"""2010-10-08""","""2019-10-31""",null,null,null,"""4bf58dd8d48988d117951735""","""8b1973341773fff""","""Retail""","""Food and Beverage Retail""","""Candy Store"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""529c4fbd498ea2069a9ef9d7""","""Mccolls Swinton""",53.492302,-1.311678,"[""Retail > Convenience Store""]",null,null,null,null,null,null,"""2013-12-02""","""2024-05-19""",null,null,null,"""4d954b0ea243a5684a65b473""","""8b1942046cacfff""","""Retail""","""Convenience Store""",null
"""5239bcd611d280918e0bdca1""","""Mccolls @ Gorsey Lane, Warring…",53.402063,-2.578719,"[""Retail > Convenience Store""]",null,"""Warrington""","""Cheshire""",null,"""England""",null,"""2013-09-18""","""2016-11-05""",null,null,null,"""4d954b0ea243a5684a65b473""","""8b1951a13463fff""","""Retail""","""Convenience Store""",null
"""569dd3f7498eb14fa68e56f3""","""Mccolls Gainsborough Road""",53.092347,-2.451545,"[""Retail > Convenience Store""]",null,null,null,null,null,null,"""2016-01-19""","""2016-04-30""",null,null,null,"""4d954b0ea243a5684a65b473""","""8b195cd9ba91fff""","""Retail""","""Convenience Store""",null


In [8]:
defunct_retailers

Brand,WikipediaListSource,SourceURL
str,str,str
"""Payless DIY""","""Defunct retail companies of th…","""https://en.wikipedia.org/wiki/…"
"""Co-operative Building, Barrow-…","""Defunct department stores of t…","""https://en.wikipedia.org/wiki/…"
"""Glyn Webb""","""Defunct discount shops (per li…","""https://en.wikipedia.org/wiki/…"
"""House of Holland""","""Defunct discount shops (per li…","""https://en.wikipedia.org/wiki/…"
"""J F Rockhey""","""Defunct department stores of t…","""https://en.wikipedia.org/wiki/…"
…,…,…
"""Austins""","""Defunct department stores of t…","""https://en.wikipedia.org/wiki/…"
"""J R Roberts Stores""","""Defunct department stores of t…","""https://en.wikipedia.org/wiki/…"
"""Mac Fisheries""","""Defunct retail companies of th…","""https://en.wikipedia.org/wiki/…"


In [4]:
import polars as pl
import geopandas as gpd
import re

defunct_retailers = pl.read_csv("../data/input_files/defunct_uk_retailers_wikipedia_full.csv")


# 1) Drop repeats on Brand column
defunct_retailers = defunct_retailers.unique(subset=["Brand"])

# 2) Remove anything in brackets () from Brand
defunct_retailers = defunct_retailers.with_columns(
    pl.col("Brand").str.replace(r"\s*\(.*?\)", "", literal=False)
)

# -----------------------------
# Assumptions from earlier:
# - defunct_retailers: pl.DataFrame with "Brand"
# - foursquare_filtnew: pl.DataFrame with "name", "lat", "lng"
# -----------------------------

# Helper to normalize for matching
def make_key(col: pl.Expr) -> pl.Expr:
    return col.str.strip_chars().str.to_lowercase()

# 0) (Optional) Clean + dedupe defunct list like before
defunct_retailers = (
    defunct_retailers
    .unique(subset=["Brand"])
    .with_columns(
        # remove bracketed suffixes e.g. "Woolworths (UK)" -> "Woolworths"
        pl.col("Brand").str.replace(r"\s*\(.*?\)", "", literal=False)
    )
)

# 1) Build normalized keys
defunct = defunct_retailers.with_columns(brand_key = make_key(pl.col("Brand")))
fs = foursquare_filtnew.with_columns(brand_key = make_key(pl.col("name")))

# 2) Find rows to remove (the removed POIs)
removed = fs.join(defunct.select("brand_key").unique(), on="brand_key", how="inner")

# 3) Top 10 brands by number of removed POIs
removed_counts = (
    removed
    .group_by("brand_key")
    .len()
    .rename({"len": "removed_poi_count"})
    .sort("removed_poi_count", descending=True)
)

top10_keys = removed_counts.select("brand_key").head(10)

# 4) Keep only removed POIs for those top 10 brands
top10_removed = removed.join(top10_keys, on="brand_key", how="inner")

# 5) Add human-readable brand back in
defunct_lookup = defunct.select(["brand_key", "Brand"]).unique()
top10_removed = (
    top10_removed
    .join(defunct_lookup, on="brand_key", how="left")
    .rename({"Brand": "defunct_brand"})
)

# 6) Ensure lat/lng present
top10_removed = top10_removed.filter(
    pl.all_horizontal(
        pl.col("lat").is_not_null(),
        pl.col("lng").is_not_null()
    )
)

# 7) Write one GeoParquet per brand of removed locations
out_dir = Path("defunct_removed_top10")
out_dir.mkdir(parents=True, exist_ok=True)

# Partition by brand
parts = top10_removed.partition_by("defunct_brand", as_dict=False)

for i, df in enumerate(parts, start=1):
    pdf = df.to_pandas()
    gdf = gpd.GeoDataFrame(
        pdf,
        geometry=gpd.points_from_xy(pdf["lng"], pdf["lat"]),
        crs="EPSG:4326"
    )
    fname = out_dir / f"topbrand{i}.geo.parquet"
    gdf.to_parquet(fname)
    print(f"Wrote {fname.name} ({len(gdf)} points)")


Wrote topbrand1.geo.parquet (66 points)
Wrote topbrand2.geo.parquet (418 points)
Wrote topbrand3.geo.parquet (64 points)
Wrote topbrand4.geo.parquet (64 points)
Wrote topbrand5.geo.parquet (119 points)
Wrote topbrand6.geo.parquet (164 points)
Wrote topbrand7.geo.parquet (31 points)
Wrote topbrand8.geo.parquet (114 points)
Wrote topbrand9.geo.parquet (41 points)
Wrote topbrand10.geo.parquet (58 points)


In [5]:
removed_counts

brand_key,removed_poi_count
str,u32
"""somerfield""",418
"""ethel austin""",164
"""rosebys""",119
"""safeway stores""",114
"""mk one""",66
…,…
"""pontings""",1
"""sweater shop""",1
"""bargain crazy""",1


In [31]:
#search for woolworths in foursquare_filtered
woolworths_fs = foursquare_filtered.filter(pl.col("name").str.contains("woolworth"))

In [32]:
woolworths_fs

fsq_place_id,name,lat,lng,address,locality,region,postcode,admin_region,post_town,po_box,country,date_created,date_refreshed,date_closed,tel,website,fsq_category_id,fsq_category_labels,h3_11
str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],str
"""4b59c5acf964a520439728e3""","""woolworths social club""",53.603172,-2.176291,null,null,null,null,null,null,null,"""GB""","""2010-01-22""","""2024-05-02""",null,null,null,"""52e81612bcbc57f1066b7a33""","[""Community and Government > Social Club""]","""8b19424f0335fff"""
